# US Firm Characteristics: Risk Overlay Applicability

**Chapter 19 - Risk Management**

A risk overlay is a rule that closes a position on something the position does
while it is held: a stop-loss when it falls a set distance below entry, a
trailing stop when it falls that distance below its own high, a time exit after
a fixed number of bars. Every one of those rules asks what the price did
*between* the moment the position was opened and the moment it would otherwise
be closed.

This case study backtests on the vectorized forward-return path. That path
holds one weight vector per rebalance and multiplies it by the realized
forward return over the whole month; it never sees a price inside the month.
The information a stop needs is therefore not merely unused here, it is absent
from the data structure the backtest runs on. Simulating a stop on it would
mean inventing an intra-month path and reporting what the invention did.

**The absence comes from the data release, not from the backtest engine**, and that is
what makes it permanent rather than a limitation someone could fund away. Read the two
declarations in `config/setup.yaml` together. `universe.identifiers` is
`anonymous_split_scoped_firm_axis`, and the note beside it records that identifiers
persist only inside each released tensor block, with no published mapping between
blocks. The observations themselves are monthly characteristic vectors. So there is no
ticker to look a daily price up against, and no continuous firm identity to look it up
along; a within-month price series for these firms cannot be bought, joined or
reconstructed. Switching this case study to an engine path would produce the same empty
table with more machinery behind it.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 美國公司特徵：風險控管層的適用性

**第 19 章－風險管理**

風險控管層是依據部位持有期間的變化而平倉的規則：價格比進場價下跌固定幅度時觸發停損；價格比持有期間自身高點下跌該幅度時觸發移動停損；經過固定根數的價格線後按時間出場。每項規則都需要知道，從開倉到原本會平倉的時刻*之間*，價格如何變動。

本案例研究使用向量化遠期報酬路徑進行回測。該路徑在每次再平衡時持有一組權重向量，並乘上整個月實現的遠期報酬；它看不到月內價格。因此停損所需的資訊並非只是此處沒有使用，而是回測所依據的資料結構根本沒有。若在此模擬停損，就得虛構月內價格路徑，再報告這段虛構路徑的結果。

**缺失源於資料發布方式，而非回測引擎**，所以這是持續存在的限制，不是追加資金就能解決的問題。請一併閱讀 `config/setup.yaml` 中的兩項宣告。`universe.identifiers` 是 `anonymous_split_scoped_firm_axis`，旁邊的註記指出，識別碼只在各個已發布的張量區塊內持續有效，區塊之間沒有公開的對應關係。觀測資料本身則是月度特徵向量。因此，既沒有可用來查詢每日價格的股票代碼，也沒有可沿時間查詢的連續公司身分；無法購買、合併或重建這些公司的月內價格序列。即使改用引擎路徑，本案例研究仍會產生同一張空表，只是背後多了些機制。
<!-- ML4T_ZH_END c0000-p000 -->

The other case studies in the book differ on exactly this point rather than on the
quality of their engineering. An engine path is available where the instrument has a
public identifier and an intraday or daily price history to go with it. Here the release
deliberately does not publish one, because anonymity is what allowed the characteristics
to be released at all.

So this notebook establishes a boundary rather than a result. It selects the
parent run the overlays would have been applied to, states which controls the
configuration declares, and registers none of them. The registry query in
section 3 is what confirms that: an empty result there is the outcome, not a
missing input.

**Learning Objectives:**
1. Select the parent run across the baseline and allocation stages
2. Decide whether a backtest path can represent a rule before configuring it
3. Separate a governance control from a validation variant that competes on Sharpe

**Book Reference:** Chapter 19, Sections 19.3-19.6, 19.8

**Prerequisites:** the Chapter 17 allocation sweep (`12_portfolio_management`),
whose runs are in `registry.db`.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
書中其他案例研究在這一點上的差別，並非工程品質的高低。有公開識別碼，且有對應盤中或每日價格歷史的金融工具，才能使用引擎路徑。此處的資料發布刻意不提供這種識別碼，因為維持匿名才讓這些特徵得以公開。

因此，本 Notebook 確立的是界線，而非績效結果。它選出原本會套用風險控管層的母執行結果，說明設定宣告了哪些控制規則，但不登錄其中任何一項。第 3 節的登錄資料庫查詢會確認這點：空結果就是本節的結論，而非輸入資料缺漏。

**學習目標：**
1. 從基準與配置階段選擇母執行結果
2. 在設定規則前，判斷回測路徑是否能表達該規則
3. 區分治理控制與以夏普比率競爭的驗證期變體

**書籍參考：**第 19 章，第 19.3–19.6、19.8 節

**先備條件：**第 17 章的配置掃描（`12_portfolio_management`），其執行結果存於 `registry.db`。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""US Firm Characteristics: Risk: Engine-Level Risk Rules."""

import json
import time
import warnings

import polars as pl

warnings.filterwarnings("ignore")

from case_studies.research import open_study
from case_studies.utils.backtest_explorer import BacktestExplorer
from case_studies.utils.backtest_loaders import (
    VECTORIZED_CASE_STUDIES,
    get_backtest_config,
    load_backtest_prices_for,
)
from case_studies.utils.backtest_presets import (
    clone_backtest_spec,
    ensure_backtest_spec,
    strategy_view,
)
from case_studies.utils.backtest_runner import precompute_weights, run_backtest
from case_studies.utils.registry import read_predictions, resolve_best_backtest_runs
from case_studies.utils.sweep_config import (
    calibrate_trailing_stops,
    get_portfolio_risk_controls,
    get_position_risk_controls,
    get_top_n_predictions,
)
from utils.paths import get_case_study_dir

`MAX_SYMBOLS` reduces the price panel and nothing else. The vectorized path takes its
universe and its P&L from the predictions frame and reads the panel only for the
rebalance calendar, so lowering it does not shrink a backtest here. It stays in the
cell because the same parameter is what reduces the engine-path case studies, and a
test harness binds it uniformly across all of them.

<!-- ML4T_ZH_BEGIN c0002-p000 -->
`MAX_SYMBOLS` 只縮減價格面板，其他資料不變。向量化路徑從預測資料框架取得股票範圍與損益，只將價格面板用於再平衡日曆，因此降低此參數不會縮小這裡的回測規模。它仍留在此儲存格，因為同一參數能縮減其他採引擎路徑的案例研究，而測試框架會在所有案例研究中以一致方式設定它。
<!-- ML4T_ZH_END c0002-p000 -->


In [2]:
CASE_STUDY_ID = "us_firm_characteristics"
LABEL = ""
MAX_SYMBOLS = 0
# Zero means all controls; a positive value limits position and portfolio
# controls each.
MAX_RISK_VARIANTS = 0
TOP_N_COMBOS = None
# Both names stay bound here although nothing below reads them: that is what makes the harness
# force preview and supply a workspace - `_declares_tier_and_workspace` in `tests/pm_helpers.py`
# looks for exactly this pair. Without them the canonical branch regenerates in place, which
# needs symlinks a CI checkout does not have.
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""

The study is opened before anything resolves a path or reads the registry. Under the preview
tier, opening it activates a workspace and rewrites `ML4T_OUTPUT_DIR` process-wide, and every
later `get_case_study_dir` call resolves against that. A `CASE_DIR`, a candidate index or a
`BacktestExplorer` built first would address the released registry while this notebook writes
to the preview one, and the two never meet.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
在解析任何路徑或讀取登錄資料庫之前，先開啟本研究。在預覽層級下，開啟研究會啟用工作區，並在整個程序範圍內改寫 `ML4T_OUTPUT_DIR`；之後每次呼叫 `get_case_study_dir` 都會相對於該工作區解析。若先建立 `CASE_DIR`、候選項目索引或 `BacktestExplorer`，它們會指向正式發布的登錄資料庫，但本 Notebook 會寫入預覽版的登錄資料庫，兩者因此無法銜接。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)

In [4]:
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
bt_config = get_backtest_config(CASE_STUDY_ID)
if TOP_N_COMBOS is None:
    TOP_N_COMBOS = get_top_n_predictions(CASE_STUDY_ID, "risk_overlay")
if not LABEL:
    LABEL = bt_config.primary_label

IS_VECTORIZED = CASE_STUDY_ID in VECTORIZED_CASE_STUDIES
MODE_LABEL = "vectorized" if IS_VECTORIZED else "engine"
print(f"Case study: {CASE_STUDY_ID}, label: {LABEL}, mode: {MODE_LABEL}")

Case study: us_firm_characteristics, label: fwd_ret_1m, mode: vectorized


## 1. The Parent Run

An overlay is applied to something, so the first step is to say what. The
candidate is drawn from two stages at once: the equal-weight baselines from
`11_backtest` and the allocator variants from `12_portfolio_management`. Taking
the higher validation Sharpe of the two rather than always taking the allocator
keeps the funnel honest in the case where portfolio construction did not improve
on the equal-weight parent it was given.

The selection runs on validation months alone, and every number below comes from
them. The holdout period stays for the strategy analysis notebook.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 1. 母執行結果

風險控管層必須套用在某個對象上，因此第一步是明確指出該對象。候選結果同時來自兩個階段：`11_backtest` 的等權重基準，以及 `12_portfolio_management` 的配置方法變體。比較兩者的驗證期夏普比率並選擇較高者，而不是一律選配置方法，能在投資組合建構未改善既有等權重母策略時，忠實呈現篩選流程。

篩選只使用驗證月份，下方所有數字也都來自這些月份。保留測試期留待策略分析 Notebook 使用。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
def _resolve_pre_risk_runs(case_study: str, label: str, *, split: str, top_n: int) -> pl.DataFrame:
    candidates = [
        resolve_best_backtest_runs(
            case_study,
            label,
            split=split,
            stage=stage,
            top_n=top_n,
        )
        for stage in ("signal", "allocation")
    ]
    candidates = [frame for frame in candidates if not frame.is_empty()]
    if not candidates:
        return pl.DataFrame()
    return (
        pl.concat(candidates)
        .sort("sharpe", descending=True)
        .unique("backtest_hash", maintain_order=True)
        .head(top_n)
    )

In [6]:
top_combos = _resolve_pre_risk_runs(
    CASE_STUDY_ID,
    LABEL,
    split="validation",
    top_n=TOP_N_COMBOS,
)

if top_combos.is_empty():
    msg = "No baseline or allocation results found. Run the upstream notebooks first."
    raise RuntimeError(msg)

for row in top_combos.iter_rows(named=True):
    spec = json.loads(row["spec_json"])
    alloc = strategy_view(spec).get("allocation", {}).get("method", "equal_weight")
    # A ruined account has no Sharpe. The engine writes NULL for every ratio metric once equity
    # reaches zero, and 29 of this case study's registered allocation runs are in that state -
    # all of them at `top_k=5`, where the long-short mapping holds ten names. At the declared
    # `top_n` the parent is always the best-ranked run, which by construction is not one of them,
    # so the format string below never met a None until the pool was widened.
    sharpe = row["sharpe"]
    shown = f"{sharpe:.3f}" if sharpe is not None else "ruined"
    print(f"  Sharpe={shown}  alloc={alloc}  bt_hash={row['backtest_hash'][:8]}")

  Sharpe=2.870  alloc=conformal_weighted  bt_hash=25e0a7c9


In [7]:
prices = load_backtest_prices_for(CASE_STUDY_ID, LABEL, split="validation", max_symbols=MAX_SYMBOLS)

### MAE/MFE-Calibrated Trailing Stops

Maximum adverse excursion is the furthest a position moved against the direction it was
opened in before it was closed; maximum favourable excursion is the furthest it moved in
that direction. Both are properties of the path a position travelled while it was held,
and calibrating a stop from them means setting the threshold where it would have avoided
the losers without cutting the winners short: a stop tighter than the typical winner's
adverse excursion closes trades that were about to work.

That calibration therefore needs the same thing the stops themselves need, which is a
price between the open and the close. On the vectorized monthly-outcome path a position
has an entry weight and a realised month, and no excursion at all - not an unmeasured
one, an undefined one. So this calibration is skipped and the configured
position-control catalog is left unexecuted.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
### 依 MAE/MFE 校準的移動停損

最大不利變動是部位在平倉前，價格朝開倉方向的反方向移動的最大幅度；最大有利變動則是朝開倉方向移動的最大幅度。兩者都是部位持有期間所經歷價格路徑的性質。依據它們校準停損，意即將門檻設在能避開虧損交易、又不會過早截斷獲利交易的位置：若停損門檻小於典型獲利交易曾經歷的不利變動，就會平掉即將奏效的交易。

因此，校準需要的資訊與停損本身相同，也就是進場與出場之間的價格。在向量化月度結果路徑上，一個部位只有進場權重與已實現的整月結果，完全沒有可定義的價格變動幅度；這不是未測量，而是未定義。因此略過此項校準，設定中宣告的部位控制清單也不執行。
<!-- ML4T_ZH_END c0011-p000 -->


In [8]:
_position_grid = get_position_risk_controls(CASE_STUDY_ID)
if not IS_VECTORIZED and "close" in prices.columns:
    calibrated = calibrate_trailing_stops(prices)
    if calibrated:
        existing_thresholds = {rc.get("threshold", 0) for rc in _position_grid}
        new_calibrated = [c for c in calibrated if c["threshold"] not in existing_thresholds]
        position_controls = _position_grid + new_calibrated
        print(f"MAE/MFE calibration added {len(new_calibrated)} thresholds")
    else:
        position_controls = _position_grid
        print("MAE/MFE calibration returned no results; using standard grid")
else:
    position_controls = _position_grid
    reason = (
        "the backtest path is vectorized"
        if IS_VECTORIZED
        else "the price panel carries no close column"
    )
    print(f"Skipping MAE/MFE calibration: {reason}")

portfolio_controls = get_portfolio_risk_controls(CASE_STUDY_ID)
# Portfolio-limit overlays were purged 2026-05-17; this CS sweeps position-level
# overlays only. Fail loudly if a portfolio overlay is ever re-introduced into
# setup.yaml so it cannot silently re-file overlay backtests against the spine.
assert not portfolio_controls, (
    f"Unexpected portfolio risk controls for {CASE_STUDY_ID}: {portfolio_controls}. "
    "Portfolio-limit overlays were removed; only position-level overlays are swept."
)
if MAX_RISK_VARIANTS > 0:
    position_controls = position_controls[:MAX_RISK_VARIANTS]
    portfolio_controls = portfolio_controls[:MAX_RISK_VARIANTS]
    print(f"Risk variants limited to {MAX_RISK_VARIANTS} each")

Skipping MAE/MFE calibration: the backtest path is vectorized


## 2. Risk Overlay Sweep

On an engine-path case study this loop registers one backtest per position-level
control. Here the position loop is skipped because the path cannot represent the
rules, and the portfolio-control list is empty by configuration, so the loop body
has nothing to register and the count below is zero by construction rather than by
failure. The two are different outcomes and the counters separate them.

The two lists are empty for different reasons, and only one of them is about this
backtest path. A position-level control asks what one position did while it was held,
so it is blocked by the missing intra-month price. A portfolio-level control asks what
the book looked like at a rebalance: gross exposure, the largest weight any single name
may carry, the number of names that must be held. Every one of those is answerable from
the weight vector this path does hold, so the vectorized path is no obstacle to them.

They are absent because `config/setup.yaml` declares none, and that is a position rather
than an oversight. A gross-exposure limit or a per-name cap is a constraint the desk
operates under whatever the backtest says, so it is specified from outside and not
discovered from the data. Sweeping it alongside the allocators would enter it into a
competition ranked on validation Sharpe, and that ranking cannot answer the question the
limit exists to settle: how much loss the desk is willing to be exposed to. Whichever cap
scored highest over the validation months would be the recommendation, and the mandate
would have been decided by an estimate rather than by the people who carry the risk.


<!-- ML4T_ZH_BEGIN c0013-p000 -->
## 2. 風險控管層掃描

在採用引擎路徑的案例研究中，此迴圈會為每項部位層級控制規則登錄一次回測。在此，因路徑無法表達這些規則，部位控制迴圈會略過；投資組合層級控制清單則依設定為空，因此迴圈主體沒有可登錄的內容，下方計數按設計為零，而不是執行失敗。兩者是不同結果，計數器會加以區分。

兩份清單為空的原因不同，只有其中一個與本回測路徑有關。部位層級控制需要知道單一部位在持有期間的變化，因此會受缺少月內價格所阻礙。投資組合層級控制則要檢視再平衡時投資組合的狀態：總曝險、單一股票可占的最大權重，以及最低持股檔數。這些問題都能從此路徑持有的權重向量回答，因此向量化路徑不會阻礙它們。

它們缺席，是因為 `config/setup.yaml` 沒有宣告任何一項；這是一種立場，而非疏忽。總曝險限制或單一股票權重上限，是交易部門不論回測結果如何都必須遵守的約束，因此應由外部指定，而不是從資料中探索。若與配置方法一同掃描，就會讓它加入按驗證期夏普比率排名的競賽；但這種排名無法回答限制原本要處理的問題：交易部門願意承擔多少損失曝險。若以驗證月份得分最高的上限作為建議，風險承擔者的授權範圍就會由估計值取代決策者來決定。
<!-- ML4T_ZH_END c0013-p000 -->

Which cap that is cannot be predicted in either direction, and it is worth being clear
that the argument does not rest on one. Scaling every weight by a constant leaves Sharpe
unchanged before costs. A per-name cap is not a scaling: `_cap_weights` in
`case_studies/utils/allocation.py` clips any weight above `max_weight` and spreads the
excess in equal parts across the names still under it, iterating until none is over. **No
name is dropped** - the holding set after the cap is the holding set before it - so what a
binding cap changes is the relative exposure across an unchanged set of names. That moves
realised Sharpe, and it can move it either way: the cap takes exposure from whichever
names the allocator weighted most heavily and gives it to the rest, which helps when the
heaviest names underperformed and hurts when they carried the return. The objection is to
the question, not to the answer it would return.

Where this case study does constrain concentration it does so through `top_k`, which is
declared in the strategy and swept as part of it.

<!-- ML4T_ZH_BEGIN c0013-p001 -->
哪一種上限會勝出，無法預先判斷；需要說清楚的是，這項論點不依賴特定方向。在計入成本前，將所有權重乘上同一常數不會改變夏普比率。單一股票權重上限並非單純縮放：`_cap_weights` 在 `case_studies/utils/allocation.py` 中會將高於 `max_weight` 的權重截斷，把超額部分平均分配給仍低於上限的股票，反覆處理直到沒有權重超標。**不會剔除任何股票**：套用上限前後，持股集合相同。因此，具約束力的上限所改變的是同一組股票之間的相對曝險。這會影響實現的夏普比率，方向可能向上或向下：上限從配置方法賦予最高權重的股票移走曝險，轉給其他股票；若最高權重股票表現不佳，就會有幫助，若它們是報酬來源，就會造成損失。反對的是所問的問題，而非可能得到的答案。

本案例研究確實會透過 `top_k` 約束集中程度；此參數在策略中宣告，並作為策略的一部分掃描。
<!-- ML4T_ZH_END c0013-p001 -->


In [9]:
n_done = 0
n_failed = 0

Every run inside the loop below is fed `combo_weights`, and computing those means
running the parent's allocator again. Where neither control list can produce a run,
that work has no consumer, so the loop is not entered at all and the weights are
never computed.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
下方迴圈中的每次執行都會接收 `combo_weights`，而計算這些值意味著要再次執行母策略的配置方法。當兩份控制清單都無法產生任何執行結果時，這項工作就沒有使用者，因此完全不進入迴圈，也不計算權重。
<!-- ML4T_ZH_END c0015-p000 -->


In [10]:
will_register = bool(portfolio_controls) or (not IS_VECTORIZED and bool(position_controls))
if not will_register:
    print(
        "No control can run on this backtest path, so no allocation weights are "
        "computed and no backtest is registered."
    )

for combo_idx, combo_row in enumerate(top_combos.iter_rows(named=True) if will_register else []):
    pred_hash = combo_row["prediction_hash"]
    base_spec = ensure_backtest_spec(
        CASE_STUDY_ID,
        bt_config,
        json.loads(combo_row["spec_json"]),
        prices=prices,
        prediction_hash=pred_hash,
        initial_cash=bt_config.initial_cash,
    )
    alloc_method = strategy_view(base_spec).get("allocation", {}).get("method", "equal_weight")

    predictions = read_predictions(CASE_STUDY_ID, pred_hash)

    t0 = time.time()
    combo_weights = precompute_weights(
        predictions, base_spec, prices, label=LABEL, case_study=CASE_STUDY_ID
    )
    print(
        f"  Combo {combo_idx + 1}/{len(top_combos)}: {alloc_method} - "
        f"weights precomputed in {time.time() - t0:.0f}s"
    )

    # Position-level risk rules (engine only)
    if not IS_VECTORIZED:
        for rc in position_controls:
            spec_risk = clone_backtest_spec(base_spec)
            spec_risk["chapter"] = "ch19"
            if rc["type"] == "time_exit":
                spec_risk["strategy"]["risk"] = {
                    "name": rc["name"],
                    "position_rules": [{"type": rc["type"], "bars": rc["bars"]}],
                }
            else:
                spec_risk["strategy"]["risk"] = {
                    "name": rc["name"],
                    "position_rules": [{"type": rc["type"], "threshold": rc["threshold"]}],
                }

            try:
                result = run_backtest(
                    CASE_STUDY_ID,
                    pred_hash,
                    spec_risk,
                    prices=prices,
                    predictions=predictions,
                    label=LABEL,
                    register=True,
                    initial_cash=bt_config.initial_cash,
                    calendar=bt_config.calendar,
                    precomputed_weights=combo_weights,
                )
                n_done += 1
                print(
                    f"    {rc['name']}: Sharpe={result.metrics.get('sharpe', 0):.3f}, "
                    f"MaxDD={result.metrics.get('max_drawdown', 0):.2%}"
                )
            except Exception as e:
                n_failed += 1
                print(f"    {rc['name']}: FAILED - {e}")

    # Portfolio-level risk limits
    for rc in portfolio_controls:
        spec_risk = clone_backtest_spec(base_spec)
        spec_risk["chapter"] = "ch19"
        spec_risk["strategy"]["risk"] = {
            "name": rc["name"],
            "portfolio_limits": [{"type": rc["type"], "threshold": rc["threshold"]}],
        }

        try:
            result = run_backtest(
                CASE_STUDY_ID,
                pred_hash,
                spec_risk,
                prices=prices,
                predictions=predictions,
                label=LABEL,
                register=True,
                initial_cash=bt_config.initial_cash,
                calendar=bt_config.calendar,
                precomputed_weights=combo_weights,
            )
            n_done += 1
            print(
                f"    {rc['name']}: Sharpe={result.metrics.get('sharpe', 0):.3f}, "
                f"MaxDD={result.metrics.get('max_drawdown', 0):.2%}"
            )
        except Exception as e:
            n_failed += 1
            print(f"    {rc['name']}: FAILED - {e}")

print(f"\nRisk sweep complete: {n_done} registered, {n_failed} failed")

No control can run on this backtest path, so no allocation weights are computed and no backtest is registered.

Risk sweep complete: 0 registered, 0 failed


## 3. What The Registry Holds

This section only reads. It asks the registry for every overlay run filed against
this case study and, for each, the change in Sharpe against the parent it was
applied to.

An empty answer here is the point of the notebook rather than a gap in it. A Sharpe
delta next to each rule would read exactly as one a stop had earned, and on this path
it could only come from an intra-month price series the data does not contain, so an
empty table is the honest form of the answer.

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 3. 登錄資料庫中的內容

本節只讀取資料。它向登錄資料庫查詢此案例研究名下的每筆風險控管層執行結果，以及各自相對於套用對象的母策略，夏普比率有何變化。

此處得到空結果，正是本 Notebook 的要點，而非缺漏。若在每條規則旁列出夏普比率變化，讀者會認為那是停損帶來的結果；但在這條路徑上，這種數值只能來自資料中不存在的月內價格序列，因此空表才是誠實的答案。
<!-- ML4T_ZH_END c0017-p000 -->


The read is scoped to the prediction the parent run carries. The registry accumulates
across labels and across earlier funnels, and this section's answer is a count of
rows, so an unscoped read would turn an overlay row filed under some other selection
into evidence about this one - which is the single way this notebook's argument could
be reported as refuted by rows that never tested it.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
讀取範圍限定於母執行結果所使用的預測。登錄資料庫會累積不同標籤與先前篩選流程的資料，而本節的答案是資料列數量；若不限制範圍，其他篩選結果下登錄的一筆風險控管層資料，就可能被誤作本次結果的證據。這是唯一可能讓本 Notebook 的論點被從未檢驗它的資料列誤報為遭到推翻的方式。
<!-- ML4T_ZH_END c0018-p000 -->


In [11]:
explorer = BacktestExplorer(CASE_STUDY_ID)
parent_hash = top_combos["prediction_hash"][0]

In [12]:
risk_df = explorer.risk_impact(prediction_hash=parent_hash)

if risk_df.is_empty():
    print("No risk overlay run is filed against the parent run, which is the outcome.")
else:
    print(f"Risk overlays filed against the parent run: {len(risk_df)}")
    with pl.Config(tbl_rows=risk_df.height):
        print(
            risk_df.select("risk_name", "risk_type", "sharpe", "max_drawdown", "sharpe_delta").sort(
                "sharpe_delta", descending=True
            )
        )

No risk overlay run is filed against the parent run, which is the outcome.


## Key Takeaways

1. Whether a rule can be represented is a property of the backtest path, not a
   setting. A stop needs a price between rebalances; the vectorized forward-return
   path holds one return per rebalance and has none, so a stop cannot be evaluated
   on it at any parameter value.
2. The configuration still declares the position-level controls, because the same
   file drives the engine-path case studies where they do run. Declared and
   applicable are separate questions, and this notebook answers the second.
3. Portfolio-level limits are absent on purpose. A gross-exposure or per-name cap
   is a constraint the desk operates under, not a variant that competes for the
   highest validation Sharpe, and sweeping it as one invites keeping whichever cap
   was loosest on the grounds that it scored highest.
4. The overlay stage registers nothing and reads no holdout month, so the funnel
   enters the strategy analysis carrying the parent run from section 1 unchanged.

**Next:** [`14_costs`](14_costs.ipynb), which sweeps the cost grid over the
configuration this stage's result is one candidate for.
[`17_strategy_analysis`](17_strategy_analysis.ipynb) confronts the selection this funnel
performed and is where the results are interpreted.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
## 重點摘要

1. 規則能否表達，取決於回測路徑的性質，而非設定。停損需要再平衡之間的價格；向量化遠期報酬路徑每次再平衡只持有一個報酬值，沒有中間價格，因此無論參數設為何值，都無法在此評估停損。
2. 設定仍宣告部位層級控制，因為同一份檔案也用於採引擎路徑、確實會執行這些控制的案例研究。已宣告與可適用是兩個不同問題，本 Notebook 回答後者。
3. 投資組合層級限制是刻意缺席的。總曝險或單一股票權重上限是交易部門必須遵守的約束，而非爭奪最高驗證期夏普比率的變體；若將它納入掃描，就可能因某個上限得分最高，而選擇限制最寬鬆的上限。
4. 風險控管層階段不登錄任何結果，也不讀取任何保留測試月份，因此篩選流程會帶著第 1 節的母執行結果，原樣進入策略分析。

**下一步：**[`14_costs`](14_costs.ipynb)，會對本階段結果也是其候選之一的組態掃描成本網格。[`17_strategy_analysis`](17_strategy_analysis.ipynb) 會處理此篩選流程所做的選擇，並解讀結果。
<!-- ML4T_ZH_END c0021-p000 -->
